#### RAG EDA

In [2]:
import os
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()

# опционально:
# os.environ['LANGCHAIN_TRACING_V2'] = 'true'
# os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'

# os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGCHAIN_API_KEY')

APP_DIR = Path.cwd().resolve().parents[1]  # .../app
RAW_PDF = APP_DIR / "app" / "data" / "raw" / "nordguard_game_tutorial.pdf"
OUT_JSONL = APP_DIR / "app" / "data" / "processed" / "docs.jsonl"
api_key = os.getenv('OPENAI_API_KEY')


print("APP_DIR:", APP_DIR)
print("RAW_PDF:", RAW_PDF, RAW_PDF.exists())
print("OUT_JSONL:", OUT_JSONL)


APP_DIR: /home/pydev/Desktop/courses/NPL_project
RAW_PDF: /home/pydev/Desktop/courses/NPL_project/app/data/raw/nordguard_game_tutorial.pdf True
OUT_JSONL: /home/pydev/Desktop/courses/NPL_project/app/data/processed/docs.jsonl


##### Part 1: Overview

In [3]:
from langchain_community.document_loaders import PyPDFLoader

# Replace 'your_file.pdf' with the path to your PDF file
pdf_loader = PyPDFLoader(RAW_PDF)

pdf_docs = pdf_loader.load()


In [ ]:
pdf_docs

#### RAG Langchain EDA

In [ ]:
from app.src.utils import save_splits_to_jsonl, clean_pdf_docs_inplace

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma


clean_pdf_docs_inplace(pdf_docs)

pdf_docs

# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
splits = text_splitter.split_documents(pdf_docs)
save_splits_to_jsonl(splits, OUT_JSONL)
print(f"✂️ Created {len(splits)} document chunks")

#### Embed and Chroma vector store

In [ ]:
from langchain_openai import OpenAIEmbeddings

from app.config import settings


embeddings = OpenAIEmbeddings( model=settings.EMBEDDING_MODEL, openai_api_key=settings.OPENAI_API_KEY)
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory=str(settings.CHROMA_DIR),
    collection_name=settings.COLLECTION_NAME,
)



#### Retrieved

In [ ]:


question = "какие ресурсы нужны для строительство большого здания?"

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 80, "lambda_mult": 0.5},
)
docs = retriever.invoke(question)

# docs = rerank(question, docs, top_k=5)

for i, d in enumerate(docs, 1):
    print(f"\n--- chunk {i} --- page={d.metadata.get('page')}")
    print(d.page_content[:400])

#### ChatPromptTemplate

In [ ]:

from langchain_core.prompts import ChatPromptTemplate

# Prompt
template = """Представь что ты эксперт по настольной играе НОРДГАРД: НОВЫЕ ЗЕМЛИ, отвечай строго по контексту. Ничего не додумывай.
Если в контексте есть разные режимы (например, игра командами), явно раздели ответ:
- Обычная игра (каждый сам за себя)
- Игра командами (если описано).

Контекст:
{context}

Вопрос: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
print("📝 Prompt template created")
prompt

##### LLM (updated to current model)

In [ ]:
# LLM (updated to current model)
from langchain_openai import ChatOpenAI


llm = ChatOpenAI(model="gpt-5-nano", temperature=0, openai_api_key=api_key)
print("🤖 LLM initialized")

#### LLM result

In [ ]:
from app.src.utils import docs_to_context

chain = prompt | llm
# Run
print("llm response:")

context = docs_to_context(docs)
result = chain.invoke({"context": context, "question": question})

print("✅ Answer:\n", result.content)